In [0]:

# AlertaBairro - Camada Ouro
# Etapa 1: Indicadores de alagamentos por subprefeitura

spark.sql("""
    CREATE OR REPLACE TABLE
    workspace.alertabairro.ouro_alagamentos_subprefeitura
    USING DELTA
    AS
    SELECT
        COALESCE(
            NULLIF(TRIM(subprefeitura), ''),
            'NAO INFORMADA'
        ) AS subprefeitura,

        COUNT(DISTINCT id_ocorrencia) AS total_ocorrencias,

        MIN(data_ocorrencia) AS primeira_ocorrencia,
        MAX(data_ocorrencia) AS ultima_ocorrencia

    FROM workspace.alertabairro.prata_alagamentos

    WHERE id_ocorrencia IS NOT NULL

    GROUP BY 1
""")

print("=== RANKING DE ALAGAMENTOS ===")

spark.sql("""
    SELECT *
    FROM workspace.alertabairro.ouro_alagamentos_subprefeitura
    ORDER BY total_ocorrencias DESC
    LIMIT 15
""").show(truncate=False)

print("=== VALIDACAO DA OURO ===")

spark.sql("""
    SELECT
        COUNT(*) AS total_subprefeituras,
        SUM(total_ocorrencias) AS total_alagamentos
    FROM workspace.alertabairro.ouro_alagamentos_subprefeitura
""").show()


=== RANKING DE ALAGAMENTOS ===
+----------------------------+-----------------+-------------------+-----------------+
|subprefeitura               |total_ocorrencias|primeira_ocorrencia|ultima_ocorrencia|
+----------------------------+-----------------+-------------------+-----------------+
|MP - SAO MIGUEL PAULISTA    |106              |2026-01-08         |2026-09-28       |
|CL - CAMPO LIMPO            |18               |2026-01-16         |2026-09-26       |
|VP - VILA PRUDENTE          |13               |2026-01-15         |2026-06-23       |
|MB - MBOI MIRIM             |12               |2026-01-08         |2026-02-26       |
|AF - ARICANDUVA/VILA FORMOSA|9                |2026-01-14         |2026-05-11       |
|IT - ITAIM PAULISTA         |9                |2026-01-28         |2026-09-14       |
|BT - BUTANTA                |8                |2026-01-27         |2026-09-13       |
|CS - CAPELA DO SOCORRO      |8                |2026-01-17         |2026-09-14       |
|MP - Sao Mi

In [0]:

# AlertaBairro - Camada Ouro
# Etapa 2: Diagnostico das subprefeituras

from pyspark.sql import functions as F

base = spark.sql(r"""
    SELECT
        subprefeitura,
        NULLIF(
            regexp_extract(
                UPPER(TRIM(subprefeitura)),
                r'^([A-Z]{2})\s*-',
                1
            ),
            ''
        ) AS codigo
    FROM workspace.alertabairro.prata_alagamentos
""")

print("=== DIAGNOSTICO ===")

base.select(
    F.count("*").alias("ocorrencias"),
    F.countDistinct("subprefeitura").alias("nomes_distintos"),
    F.countDistinct("codigo").alias("codigos_distintos"),
    F.sum(
        F.col("codigo").isNull().cast("int")
    ).alias("registros_sem_codigo")
).show()

print("=== NOMES COM GRAFIAS DIFERENTES ===")

(
    base.groupBy("codigo")
    .agg(
        F.count("*").alias("ocorrencias"),
        F.countDistinct("subprefeitura").alias("grafias"),
        F.collect_set("subprefeitura").alias("nomes_encontrados")
    )
    .filter(
        (F.col("grafias") > 1) |
        F.col("codigo").isNull()
    )
    .orderBy(F.desc("grafias"))
    .show(40, truncate=False)
)


=== DIAGNOSTICO ===
+-----------+---------------+-----------------+--------------------+
|ocorrencias|nomes_distintos|codigos_distintos|registros_sem_codigo|
+-----------+---------------+-----------------+--------------------+
|        292|             46|               29|                   1|
+-----------+---------------+-----------------+--------------------+

=== NOMES COM GRAFIAS DIFERENTES ===
+------+-----------+-------+------------------------------------------------------------+
|codigo|ocorrencias|grafias|nomes_encontrados                                           |
+------+-----------+-------+------------------------------------------------------------+
|PI    |3          |2      |[PI - Pinheiros, PI - PINHEIROS]                            |
|VP    |15         |2      |[VP - Vila Prudente, VP - VILA PRUDENTE]                    |
|CV    |4          |2      |[CV - CASA VERDE/CACHOEIRINHA, CV - Casa Verde/Cachoeirinha]|
|JA    |4          |2      |[JA - Jabaquara, JA - JABAQUA

In [0]:

# AlertaBairro - Camada Ouro
# Etapa 3: Padronizar subprefeituras

spark.sql(r"""
    CREATE OR REPLACE TABLE
    workspace.alertabairro.ouro_alagamentos_subprefeitura
    USING DELTA
    AS

    WITH base AS (
        SELECT
            id_ocorrencia,
            data_ocorrencia,

            UPPER(TRIM(subprefeitura)) AS nome_normalizado,

            NULLIF(
                REGEXP_EXTRACT(
                    UPPER(TRIM(subprefeitura)),
                    r'^([A-Z]{1,2})\s*-',
                    1
                ),
                ''
            ) AS codigo

        FROM workspace.alertabairro.prata_alagamentos
        WHERE id_ocorrencia IS NOT NULL
    ),

    padronizada AS (
        SELECT
            *,
            COALESCE(
                codigo,
                CONCAT('SEM_CODIGO:', nome_normalizado)
            ) AS chave
        FROM base
    )

    SELECT
        MAX(codigo) AS codigo_subprefeitura,
        MIN(nome_normalizado) AS subprefeitura,
        COUNT(DISTINCT id_ocorrencia) AS total_ocorrencias,
        MIN(data_ocorrencia) AS primeira_ocorrencia,
        MAX(data_ocorrencia) AS ultima_ocorrencia

    FROM padronizada
    GROUP BY chave
""")

print("=== RANKING PADRONIZADO ===")

spark.sql("""
    SELECT
        codigo_subprefeitura,
        subprefeitura,
        total_ocorrencias
    FROM workspace.alertabairro.ouro_alagamentos_subprefeitura
    ORDER BY total_ocorrencias DESC
    LIMIT 15
""").show(truncate=False)

print("=== VALIDACAO FINAL ===")

spark.sql("""
    SELECT
        COUNT(*) AS total_grupos,
        SUM(total_ocorrencias) AS total_alagamentos,
        SUM(
            CASE WHEN codigo_subprefeitura IS NULL
            THEN 1 ELSE 0 END
        ) AS grupos_sem_codigo
    FROM workspace.alertabairro.ouro_alagamentos_subprefeitura
""").show()


=== RANKING PADRONIZADO ===
+--------------------+----------------------------+-----------------+
|codigo_subprefeitura|subprefeitura               |total_ocorrencias|
+--------------------+----------------------------+-----------------+
|MP                  |MP - SAO MIGUEL PAULISTA    |113              |
|CL                  |CL - CAMPO LIMPO            |24               |
|MB                  |MB - MBOI MIRIM             |16               |
|AF                  |AF - ARICANDUVA/VILA FORMOSA|16               |
|VP                  |VP - VILA PRUDENTE          |15               |
|BT                  |BT - BUTANTA                |11               |
|CS                  |CS - CAPELA DO SOCORRO      |9                |
|IT                  |IT - ITAIM PAULISTA         |9                |
|IP                  |IP - IPIRANGA               |9                |
|PJ                  |PJ - PIRITUBA/JARAGUA       |7                |
|JT                  |JT - JACANA/TREMEMBE        |6          

In [0]:

# AlertaBairro - Camada Ouro
# Etapa 4: Evolucao mensal dos alagamentos

spark.sql("""
    CREATE OR REPLACE TABLE
    workspace.alertabairro.ouro_alagamentos_mensal
    USING DELTA
    AS

    WITH ocorrencias AS (
        SELECT
            id_ocorrencia,

            CAST(
                DATE_TRUNC('MONTH', data_ocorrencia)
                AS DATE
            ) AS mes

        FROM workspace.alertabairro.prata_alagamentos

        WHERE id_ocorrencia IS NOT NULL
          AND data_ocorrencia IS NOT NULL
    )

    SELECT
        mes,
        COUNT(DISTINCT id_ocorrencia) AS total_ocorrencias

    FROM ocorrencias
    GROUP BY mes
""")

print("=== EVOLUCAO MENSAL DOS ALAGAMENTOS ===")

spark.sql("""
    SELECT
        mes,
        total_ocorrencias

    FROM workspace.alertabairro.ouro_alagamentos_mensal

    ORDER BY mes
""").show(20, truncate=False)

print("=== VALIDACAO DA OURO MENSAL ===")

spark.sql("""
    SELECT
        COUNT(*) AS meses_com_ocorrencias,
        SUM(total_ocorrencias) AS total_alagamentos,
        MIN(mes) AS primeiro_mes,
        MAX(mes) AS ultimo_mes

    FROM workspace.alertabairro.ouro_alagamentos_mensal
""").show()


=== EVOLUCAO MENSAL DOS ALAGAMENTOS ===
+----------+-----------------+
|mes       |total_ocorrencias|
+----------+-----------------+
|2026-01-01|94               |
|2026-02-01|34               |
|2026-03-01|48               |
|2026-04-01|5                |
|2026-05-01|1                |
|2026-06-01|10               |
|2026-08-01|5                |
|2026-09-01|95               |
+----------+-----------------+

=== VALIDACAO DA OURO MENSAL ===
+---------------------+-----------------+------------+----------+
|meses_com_ocorrencias|total_alagamentos|primeiro_mes|ultimo_mes|
+---------------------+-----------------+------------+----------+
|                    8|              292|  2026-01-01|2026-09-01|
+---------------------+-----------------+------------+----------+



In [0]:

# AlertaBairro - Camada Ouro
# Etapa 5: Indicadores climaticos diarios

spark.sql("""
    CREATE OR REPLACE TABLE
    workspace.alertabairro.ouro_clima_diario
    USING DELTA
    AS

    SELECT
        data,

        precipitacao_mm,

        ROUND(
            SUM(precipitacao_mm) OVER (
                ORDER BY data
                ROWS BETWEEN UNBOUNDED PRECEDING
                AND CURRENT ROW
            ),
            2
        ) AS precipitacao_acumulada_mm

    FROM workspace.alertabairro.prata_clima

    WHERE data IS NOT NULL
      AND precipitacao_mm IS NOT NULL
      AND precipitacao_mm >= 0
""")

print("=== PRECIPITACAO DIARIA ===")

spark.sql("""
    SELECT *
    FROM workspace.alertabairro.ouro_clima_diario
    ORDER BY data
""").show(20, truncate=False)

print("=== INDICADORES CLIMATICOS ===")

spark.sql("""
    SELECT
        COUNT(*) AS total_dias,
        COUNT(DISTINCT data) AS datas_unicas,
        ROUND(SUM(precipitacao_mm), 2) AS chuva_total_mm,
        MAX(precipitacao_mm) AS maior_chuva_diaria_mm,
        MIN(data) AS primeira_data,
        MAX(data) AS ultima_data
    FROM workspace.alertabairro.ouro_clima_diario
""").show()


=== PRECIPITACAO DIARIA ===
+----------+---------------+-------------------------+
|data      |precipitacao_mm|precipitacao_acumulada_mm|
+----------+---------------+-------------------------+
|2026-10-28|1.6            |1.6                      |
|2026-10-29|20.2           |21.8                     |
|2026-10-30|5.2            |27.0                     |
|2026-10-31|1.7            |28.7                     |
|2026-11-01|17.8           |46.5                     |
|2026-11-02|6.9            |53.4                     |
|2026-11-03|0.0            |53.4                     |
|2026-11-04|2.3            |55.7                     |
|2026-11-05|3.1            |58.8                     |
|2026-11-06|8.3            |67.1                     |
+----------+---------------+-------------------------+

=== INDICADORES CLIMATICOS ===
+----------+------------+--------------+---------------------+-------------+-----------+
|total_dias|datas_unicas|chuva_total_mm|maior_chuva_diaria_mm|primeira_data|ultim

In [0]:
%sql
SHOW TABLES IN workspace.alertabairro;

database,tableName,isTemporary
alertabairro,bronze_alagamentos,false
alertabairro,bronze_clima,false
alertabairro,ouro_alagamentos_mensal,false
alertabairro,ouro_alagamentos_subprefeitura,false
alertabairro,ouro_clima_diario,false
alertabairro,prata_alagamentos,false
alertabairro,prata_clima,false


In [0]:

%sql
DESCRIBE TABLE workspace.alertabairro.prata_alagamentos;


col_name,data_type,comment
id_ocorrencia,string,null
data_ocorrencia,date,null
tipo_ocorrencia,string,null
distrito,string,null
subprefeitura,string,null
longitude,double,null
latitude,double,null
data_carga_origem,date,null
# Clustering Information,,
# col_name,data_type,comment


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.alertabairro.ouro_dim_subprefeitura
USING DELTA
AS
SELECT
    UPPER(TRIM(subprefeitura)) AS id_subprefeitura,
    MIN(TRIM(subprefeitura)) AS nome_subprefeitura
FROM workspace.alertabairro.prata_alagamentos
WHERE subprefeitura IS NOT NULL
  AND TRIM(subprefeitura) <> ''
GROUP BY UPPER(TRIM(subprefeitura));


num_affected_rows,num_inserted_rows


In [0]:

%sql
SELECT
    COUNT(*) AS total_subprefeituras,
    COUNT(DISTINCT id_subprefeitura) AS ids_unicos,
    SUM(
        CASE
            WHEN id_subprefeitura IS NULL THEN 1
            ELSE 0
        END
    ) AS ids_nulos
FROM workspace.alertabairro.ouro_dim_subprefeitura;


total_subprefeituras,ids_unicos,ids_nulos
30,30,0


In [0]:

%sql
CREATE OR REPLACE TABLE workspace.alertabairro.ouro_dim_tempo
USING DELTA
AS
SELECT DISTINCT
    data_ocorrencia AS id_data,
    YEAR(data_ocorrencia) AS ano,
    MONTH(data_ocorrencia) AS mes,
    QUARTER(data_ocorrencia) AS trimestre,
    DAY(data_ocorrencia) AS dia,
    DATE_FORMAT(data_ocorrencia, 'yyyy-MM') AS ano_mes,
    DAYOFWEEK(data_ocorrencia) AS dia_semana
FROM workspace.alertabairro.prata_alagamentos
WHERE data_ocorrencia IS NOT NULL;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS total_datas,
    COUNT(DISTINCT id_data) AS datas_unicas,
    COUNT_IF(id_data IS NULL) AS datas_nulas,
    MIN(id_data) AS primeira_data,
    MAX(id_data) AS ultima_data
FROM workspace.alertabairro.ouro_dim_tempo;

total_datas,datas_unicas,datas_nulas,primeira_data,ultima_data
71,71,0,2026-01-04,2026-09-28


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.alertabairro.ouro_fato_alagamentos
USING DELTA
AS
SELECT
    id_ocorrencia,
    data_ocorrencia AS id_data,
    UPPER(TRIM(subprefeitura)) AS id_subprefeitura,
    tipo_ocorrencia,
    distrito,
    latitude,
    longitude,
    data_carga_origem,
    1 AS quantidade_ocorrencias
FROM workspace.alertabairro.prata_alagamentos
WHERE id_ocorrencia IS NOT NULL
  AND data_ocorrencia IS NOT NULL
  AND subprefeitura IS NOT NULL
  AND TRIM(subprefeitura) <> '';

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS total_ocorrencias,
    COUNT(DISTINCT id_ocorrencia) AS ids_unicos,
    COUNT_IF(id_ocorrencia IS NULL) AS ids_nulos,
    SUM(quantidade_ocorrencias) AS total_medida
FROM workspace.alertabairro.ouro_fato_alagamentos;

total_ocorrencias,ids_unicos,ids_nulos,total_medida
292,292,0,292


In [0]:
%sql
SELECT
    COUNT(*) AS total_ocorrencias,
    COUNT_IF(t.id_data IS NULL) AS sem_dim_tempo,
    COUNT_IF(s.id_subprefeitura IS NULL) AS sem_dim_subprefeitura
FROM workspace.alertabairro.ouro_fato_alagamentos f
LEFT JOIN workspace.alertabairro.ouro_dim_tempo t
    ON f.id_data = t.id_data
LEFT JOIN workspace.alertabairro.ouro_dim_subprefeitura s
    ON f.id_subprefeitura = s.id_subprefeitura;

total_ocorrencias,sem_dim_tempo,sem_dim_subprefeitura
292,0,0


In [0]:
%sql
SELECT
    s.nome_subprefeitura,
    SUM(f.quantidade_ocorrencias) AS total_alagamentos
FROM workspace.alertabairro.ouro_fato_alagamentos f
INNER JOIN workspace.alertabairro.ouro_dim_subprefeitura s
    ON f.id_subprefeitura = s.id_subprefeitura
GROUP BY s.nome_subprefeitura
ORDER BY total_alagamentos DESC
LIMIT 10;

nome_subprefeitura,total_alagamentos
MP - SAO MIGUEL PAULISTA,113
CL - CAMPO LIMPO,24
AF - ARICANDUVA/VILA FORMOSA,16
MB - MBOI MIRIM,16
VP - VILA PRUDENTE,15
BT - BUTANTA,11
IP - IPIRANGA,9
CS - CAPELA DO SOCORRO,9
IT - ITAIM PAULISTA,9
PJ - PIRITUBA/JARAGUA,7


In [0]:
%sql
SELECT
    t.ano,
    t.mes,
    t.ano_mes,
    SUM(f.quantidade_ocorrencias) AS total_alagamentos
FROM workspace.alertabairro.ouro_fato_alagamentos f
INNER JOIN workspace.alertabairro.ouro_dim_tempo t
    ON f.id_data = t.id_data
GROUP BY
    t.ano,
    t.mes,
    t.ano_mes
ORDER BY t.ano, t.mes;

ano,mes,ano_mes,total_alagamentos
2026,1,2026-01,94
2026,2,2026-02,34
2026,3,2026-03,48
2026,4,2026-04,5
2026,5,2026-05,1
2026,6,2026-06,10
2026,8,2026-08,5
2026,9,2026-09,95


In [0]:
%sql
SELECT
    t.ano,
    t.mes,
    t.ano_mes,
    SUM(f.quantidade_ocorrencias) AS total_alagamentos
FROM workspace.alertabairro.ouro_fato_alagamentos f
INNER JOIN workspace.alertabairro.ouro_dim_tempo t
    ON f.id_data = t.id_data
GROUP BY
    t.ano,
    t.mes,
    t.ano_mes
ORDER BY t.ano, t.mes;

ano,mes,ano_mes,total_alagamentos
2026,1,2026-01,94
2026,2,2026-02,34
2026,3,2026-03,48
2026,4,2026-04,5
2026,5,2026-05,1
2026,6,2026-06,10
2026,8,2026-08,5
2026,9,2026-09,95


In [0]:
%sql
WITH contagem AS (
    SELECT
        COALESCE(
            NULLIF(TRIM(tipo_ocorrencia), ''),
            'Não informado'
        ) AS tipo_ocorrencia,
        COUNT(1) AS total
    FROM workspace.alertabairro.ouro_fato_alagamentos
    GROUP BY 1
)
SELECT
    tipo_ocorrencia,
    total,
    ROUND(
        100.0 * total / SUM(total) OVER (),
        2
    ) AS percentual
FROM contagem
ORDER BY total DESC;

tipo_ocorrencia,total,percentual
ALAGAMENTO,292,100.00


In [0]:
%sql
SELECT
    CASE t.dia_semana
        WHEN 1 THEN 'Domingo'
        WHEN 2 THEN 'Segunda-feira'
        WHEN 3 THEN 'Terça-feira'
        WHEN 4 THEN 'Quarta-feira'
        WHEN 5 THEN 'Quinta-feira'
        WHEN 6 THEN 'Sexta-feira'
        WHEN 7 THEN 'Sábado'
    END AS dia_semana,
    COUNT(1) AS total_alagamentos
FROM workspace.alertabairro.ouro_fato_alagamentos f
INNER JOIN workspace.alertabairro.ouro_dim_tempo t
    ON f.id_data = t.id_data
GROUP BY t.dia_semana
ORDER BY total_alagamentos DESC;

dia_semana,total_alagamentos
Quinta-feira,49
Segunda-feira,49
Domingo,48
Terça-feira,39
Quarta-feira,39
Sábado,35
Sexta-feira,33


In [0]:
%sql
WITH analise_regional AS (
    SELECT
        s.nome_subprefeitura,
        SUM(f.quantidade_ocorrencias)
            AS total_ocorrencias,
        COUNT(DISTINCT t.ano_mes)
            AS meses_com_registros
    FROM workspace.alertabairro.ouro_fato_alagamentos f
    INNER JOIN workspace.alertabairro.ouro_dim_subprefeitura s
        ON f.id_subprefeitura = s.id_subprefeitura
    INNER JOIN workspace.alertabairro.ouro_dim_tempo t
        ON f.id_data = t.id_data
    GROUP BY s.nome_subprefeitura
)
SELECT
    nome_subprefeitura,
    total_ocorrencias,
    meses_com_registros,
    ROUND(
        100.0 * total_ocorrencias /
        SUM(total_ocorrencias) OVER (),
        2
    ) AS percentual_total
FROM analise_regional
ORDER BY
    total_ocorrencias DESC,
    meses_com_registros DESC
LIMIT 10;

nome_subprefeitura,total_ocorrencias,meses_com_registros,percentual_total
MP - SAO MIGUEL PAULISTA,113,5,38.70
CL - CAMPO LIMPO,24,5,8.22
AF - ARICANDUVA/VILA FORMOSA,16,5,5.48
MB - MBOI MIRIM,16,3,5.48
VP - VILA PRUDENTE,15,4,5.14
BT - BUTANTA,11,7,3.77
CS - CAPELA DO SOCORRO,9,4,3.08
IP - IPIRANGA,9,3,3.08
IT - ITAIM PAULISTA,9,2,3.08
PJ - PIRITUBA/JARAGUA,7,2,2.40


In [0]:
%sql
DESCRIBE TABLE workspace.alertabairro.ouro_fato_alagamentos;

col_name,data_type,comment
id_ocorrencia,string,null
id_data,date,null
id_subprefeitura,string,null
tipo_ocorrencia,string,null
distrito,string,null
latitude,double,null
longitude,double,null
data_carga_origem,date,null
quantidade_ocorrencias,int,null
# Clustering Information,,


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.alertabairro.ouro_fato_alagamentos_publica
USING DELTA
AS
SELECT
    SHA2(id_ocorrencia, 256) AS id_ocorrencia_hash,
    id_data,
    id_subprefeitura,
    tipo_ocorrencia,
    distrito,
    quantidade_ocorrencias
FROM workspace.alertabairro.ouro_fato_alagamentos;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(1) AS total_registros,
    COUNT(DISTINCT id_ocorrencia_hash) AS hashes_unicos,
    COUNT_IF(id_ocorrencia_hash IS NULL) AS hashes_nulos,
    SUM(quantidade_ocorrencias) AS total_ocorrencias
FROM workspace.alertabairro.ouro_fato_alagamentos_publica;

total_registros,hashes_unicos,hashes_nulos,total_ocorrencias
292,292,0,292


In [0]:
%sql
SELECT
    COUNT(1) AS total_registros,
    COUNT(DISTINCT id_ocorrencia_hash) AS hashes_unicos,
    COUNT_IF(id_ocorrencia_hash IS NULL) AS hashes_nulos,
    SUM(quantidade_ocorrencias) AS total_ocorrencias
FROM workspace.alertabairro.ouro_fato_alagamentos_publica;

total_registros,hashes_unicos,hashes_nulos,total_ocorrencias
292,292,0,292


In [0]:
%sql
DESCRIBE TABLE workspace.alertabairro.ouro_fato_alagamentos_publica;

col_name,data_type,comment
id_ocorrencia_hash,string,null
id_data,date,null
id_subprefeitura,string,null
tipo_ocorrencia,string,null
distrito,string,null
quantidade_ocorrencias,int,null
# Clustering Information,,
# col_name,data_type,comment
